# Importance Sampling for a Barrier Option

Down-and-out call with discrete monitoring, payoff $(S_T-K)^+\mathbf 1_{\{S_{t_j}\ge L,\ 0\le j\le J\}}$, with $S_0 = 100$, $K = 110$, $L = 80$, $r = 0.05$, $\sigma = 0.2$, $T = 2$, $J = 24$. By Cameron-Martin,
$$\mathbb E[f(W)] = \mathbb E\big[f\big((W_t + \lambda t)_{t\le T}\big)\,e^{-\lambda W_T - \lambda^2 T/2}\big],$$
and $\lambda$ is chosen to minimize $u(\lambda) = \frac{\lambda^2 T}{2} + \log \mathbb E\big[e^{-\lambda W_T} f(W)^2\big]$, which is convex with $u'' \ge T$.

In [1]:
import numpy as np

S0, K, L, r, sigma, T, J = 100.0, 110.0, 80.0, 0.05, 0.2, 2.0, 24
t = np.linspace(0, T, J + 1)
rng = np.random.default_rng(0)

def brownian(n, rng):
    increments = np.sqrt(T / J) * rng.standard_normal((n, J))
    return np.concatenate([np.zeros((n, 1)), increments.cumsum(axis=1)], axis=1)

def barrier_payoff(W):
    S = S0 * np.exp((r - 0.5 * sigma**2) * t + sigma * W)
    return np.exp(-r * T) * np.maximum(S[:, -1] - K, 0.0) * np.all(S >= L, axis=1)

## Choosing $\lambda$

Newton's method on the empirical version of $u$, computed on a pilot sample of $n = 1000$ paths.

In [2]:
W_pilot = brownian(1000, rng)
x, f2 = W_pilot[:, -1], barrier_payoff(W_pilot)**2

def weights(lam):
    w = np.exp(-lam * x) * f2
    return w / w.sum()

def u_prime(lam):
    return lam * T - weights(lam) @ x

def u_second(lam):
    w = weights(lam)
    return T + w @ x**2 - (w @ x)**2

lam = 0.0
for _ in range(5):
    lam -= u_prime(lam) / u_second(lam)
    print(f"lambda = {lam:.6f}")
lam_star = lam

lambda = 0.915866
lambda = 0.920257
lambda = 0.920257
lambda = 0.920257
lambda = 0.920257


## Comparison with crude Monte Carlo

In [3]:
M = 100_000

def summary(X):
    return X.mean(), 1.96 * X.std(ddof=1) / np.sqrt(len(X))

W = brownian(M, rng)
crude = summary(barrier_payoff(W))

W = brownian(M, rng)
importance = summary(barrier_payoff(W + lam_star * t) * np.exp(-lam_star * W[:, -1] - 0.5 * lam_star**2 * T))

print(f"Crude MC            {crude[0]:.4f} ± {crude[1]:.4f}")
print(f"Importance sampling {importance[0]:.4f} ± {importance[1]:.4f}")
print(f"Variance ratio      {(crude[1] / importance[1])**2:.1f}")

Crude MC            11.3111 ± 0.1223
Importance sampling 11.2371 ± 0.0371
Variance ratio      10.9
